# Stochastic Goose — benchmark sweep (Colab, OFFLINE)

Runs the **Stochastic Goose** replication (Dries Smit / Tufa Labs, 1st place ARC-AGI-3 preview,
12.58%) on our 4 games **offline** (local env_files — no API key/internet needed), logging the
action-step at which each level is cleared. This puts goose on the **same metric as our methods**
(env-steps to clear L1/L2/L3) so it can be overlaid on the headline staircase.

Goose = a per-level **frame-change predictor** (CNN → BCE: "will this action change the screen?"),
sampling actions ∝ predicted-change-prob; model+buffer reset each level. NOT RL/RND — a different
exploration philosophy, which is the point of the comparison.

> Set **Runtime ▸ GPU**. Goose is ~step-rate-limited by its per-step CNN; GPU helps.

## 1. Setup (clone + install; OFFLINE needs no API key)

In [ ]:
import os, sys, time, json, glob
REPO_URL = "https://github.com/LavetteSinsora/ProjectArceus.git"; REPO = "/content/ProjectArceus"
if not os.path.isdir(REPO):
    !git clone --depth 1 $REPO_URL $REPO
%cd /content/ProjectArceus
!git pull --ff-only -q || true
!pip -q install "arc-agi>=0.9.8" "arcengine>=0.9.3" torch
sys.path.insert(0, "/content/ProjectArceus/replication/card_stochastic_goose")
import torch; print("CUDA:", torch.cuda.is_available())

## 2. Config

In [ ]:
GAMES = ["ls20", "tu93", "re86", "g50t"]
SEEDS = [0, 1]                       # goose reseeds per game; re-run for variance
MAX_ACTIONS = {"ls20": 200_000, "tu93": 600_000, "re86": 1_000_000, "g50t": 300_000}
MAX_MINUTES_PER_RUN = 120            # wall-clock safety cap per (game,seed)
OUT = "/content/goose_results"; os.makedirs(OUT, exist_ok=True)
print("games", GAMES, "seeds", SEEDS)

## 3. Run goose offline, log step-at-each-level-clear

Re-runnable; writes one `result.json` per (game, seed) into `OUT`. `level_steps` maps level→action-step
of first clear (directly comparable to our `env_steps_to_first_reward`).

In [ ]:
from arc_agi import Arcade, OperationMode
from arcengine import GameAction, GameState
from agent import Action

def run_goose(game, seed, max_actions, max_minutes):
    arc = Arcade(operation_mode=OperationMode.OFFLINE)
    gid = next((e.game_id for e in arc.get_environments() if e.game_id.startswith(game)), game)
    env = arc.make(gid)
    frame = env.observation_space or env.step(GameAction.RESET)
    torch.manual_seed(seed)
    ag = Action(game_id=gid, max_minutes=max_minutes)
    level_steps, steps, t0, last_lvl = {}, 0, time.time(), -1
    while steps < max_actions and (time.time() - t0) < max_minutes * 60:
        if frame.state is GameState.WIN:
            break
        a = ag.choose_action([frame], frame)
        frame = env.step(a, data={"x": int(a.action_data.x), "y": int(a.action_data.y)}) if a == GameAction.ACTION6 else env.step(a)
        if frame is None:
            continue
        steps += 1
        lvl = getattr(frame, "levels_completed", last_lvl)
        if lvl is not None and lvl > last_lvl:
            for L in range(last_lvl + 1, lvl + 1):
                level_steps[L] = steps                      # action-step at which level L was cleared
            last_lvl = lvl
            print(f"  [{game} s{seed}] cleared up to level {lvl} @ step {steps}")
        if steps % 5000 == 0:
            print(f"  [{game} s{seed}] step {steps} levels={last_lvl} fps={steps/(time.time()-t0):.1f}")
    res = {"agent": "stochastic_goose", "game": game, "game_id": gid, "seed": seed,
           "levels_completed": last_lvl, "total_actions": steps,
           "level_steps": level_steps, "wall_seconds": time.time() - t0}
    json.dump(res, open(f"{OUT}/goose_{game}_seed{seed}.json", "w"), indent=2)
    print(f"[{game} s{seed}] DONE levels={last_lvl} steps={steps}  level_steps={level_steps}")
    return res

for g in GAMES:
    for s in SEEDS:
        run_goose(g, s, MAX_ACTIONS[g], MAX_MINUTES_PER_RUN)

## 4. Aggregate — steps-to-clear-each-level (vs our methods)

In [ ]:
import numpy as np
from collections import defaultdict
rows = [json.load(open(f)) for f in glob.glob(f"{OUT}/goose_*.json")]
agg = defaultdict(lambda: defaultdict(list))
for r in rows:
    for L, st in r["level_steps"].items():
        agg[r["game"]][int(L)].append(st)
print(f"{'game':>5} | steps to clear L1 / L2 / L3 (median over seeds; — = not cleared)")
for g in GAMES:
    cells = []
    for L in (1, 2, 3):
        v = agg[g].get(L, [])
        cells.append(f"{np.median(v):,.0f}" if v else "—")
    print(f"{g:>5} | {cells[0]:>10} {cells[1]:>10} {cells[2]:>10}")

## 5. Download

In [ ]:
import shutil
shutil.make_archive("/content/goose_results", "zip", OUT)
try:
    from google.colab import files; files.download("/content/goose_results.zip")
except Exception as e:
    print("download from Files pane:", e)